# 03 임베딩과 직접 검색

낱말이 없어도 뜻을 좌표로 바꿔 가깝기를 잰다. 점수는 절대값이 아니라 순위다.


In [ ]:
from pathlib import Path  # 경로를 문자열 대신 객체로 다룬다
import os  # 환경변수(OPENAI_API_KEY)를 넣기 위해 쓴다

# 수업 코드는 키가 이미 있는 상태를 가정한다. 이 노트북은 .env를 직접 읽는다.
for _env in (Path("../.env"), Path("../../c3-api/.env")):  # 프로젝트 루트, 옆 폴더 순으로 찾는다
    if not _env.is_file():  # 파일이 없으면 다음 후보
        continue  # 있는 파일만 읽는다
    for _line in _env.read_text(encoding="utf-8").splitlines():  # .env를 한 줄씩
        _line = _line.strip()  # 앞뒤 공백 제거
        if not _line or _line.startswith("#") or "=" not in _line:  # 빈 줄·주석·형식 아닌 줄
            continue  # 건너뛴다
        _k, _v = _line.split("=", 1)  # KEY=VALUE 로 나눈다
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))  # 이미 있으면 덮지 않는다


In [ ]:
import json  # chunks.jsonl 복구
from pathlib import Path  # 파일 위치
import time  # 임베딩 배치가 얼마나 걸리는지

import numpy as np  # 벡터 연산. 코사인 유사도는 정규화 후 내적
from openai import OpenAI  # 임베딩 API

client = OpenAI()  # OPENAI_API_KEY 사용
EMBED_MODEL = "text-embedding-3-small"  # 수업에서 쓰는 임베딩 모델
API_MODEL = "gpt-5.6-luna"  # 생성은 나중에 쓸 수 있게 이름만 둔다

# 청킹된 jsonl 파일을 각 줄을 파이썬 객체화 시켜서 복구
chunks = []  # 빈 리스트
for line in Path("chunks.jsonl").read_text(encoding="utf-8").splitlines():  # 한 줄 = 조각 하나
    chunks.append(json.loads(line))  # 문자열 JSON → dict
print("조각", len(chunks))  # 02 에서 저장한 개수와 같아야 한다


In [ ]:
QUESTION = "어린이 정보를 다룰 때 지켜야 할 것은?"  # 키워드 검색으로 한계를 보여 줄 질문
print("질문:", QUESTION)  # 질문을 먼저 찍는다
print()  # 빈 줄

for word in ("어린이", "다룰", "지켜야"):  # 질문에서 뽑은 낱말 세 개
    n = sum(1 for c in chunks if word in c["text"])  # 그 글자가 본문에 있는 조각 수
    print(f"  '{word}' 가 든 조각: {n:>4}개")  # 낱말 검색은 동어반복에 약하다


비슷한 문장끼리 숫자가 크고, 점심 김치찌개는 동떨어져야 한다.


In [ ]:
samples = [  # 뜻을 비교할 문장 다섯 개
    "어린이 정보를 다룰 때 지켜야 할 것은?",  # 질문
    "만 14세 미만 아동의 개인정보 보호",  # 같은 뜻, 낱말은 다름
    "정보주체는 자신의 개인정보 열람을 요구할 수 있다",  # 같은 법이지만 다른 조항
    "밀 재배 의사결정 지원 시스템은 기상 데이터를 사용한다",  # 다른 문서
    "점심은 김치찌개였다",  # 코퍼스와 무관한 문장
]
res = client.embeddings.create(model=EMBED_MODEL, input=samples)  # 다섯 문장을 한 번에 좌표로
vectors = [d.embedding for d in res.data]  # API 응답에서 숫자 리스트만 꺼낸다

V = np.array(vectors)  # 파이썬 리스트 → 행렬 (5, 차원)
V = V / np.linalg.norm(V, axis=1, keepdims=True)  # 각 행의 길이를 1로. 내적이 곧 코사인
sim = V @ V.T  # 모든 문장 쌍의 코사인 유사도 (5, 5)

for i, s in enumerate(samples):  # 번호와 문장을 같이 보여 준다
    print(f"{i}: {s}")  # 표의 축 설명
print()  # 빈 줄
print("     " + "".join(f"{j:>7}" for j in range(len(samples))))  # 열 번호
for i in range(len(samples)):  # 행
    print(f"{i}  " + "".join(f"{sim[i][j]:7.2f}" for j in range(len(samples))))  # 0~1 사이 소수


In [ ]:
import matplotlib.pyplot as plt  # 표를 색으로 그린다

plt.rcParams["font.family"] = "NanumGothic"  # 강의 코드. 실습실 윈도우 글꼴
from matplotlib import font_manager as _fm  # 이 PC에 깔린 글꼴 이름을 본다
_names = {f.name for f in _fm.fontManager.ttflist}  # 설치된 글꼴 집합
for _cand in ("NanumGothic", "Noto Sans CJK KR", "Malgun Gothic"):  # 리눅스·윈도우 후보
    if _cand in _names:  # 있으면
        plt.rcParams["font.family"] = _cand  # 그 글꼴로 바꾼다
        break  # 첫 번째로 되는 것만
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 부호가 깨지지 않게

fig, ax = plt.subplots(figsize=(7, 5.5))  # 그림과 좌표축
im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)  # 0 은 연한색, 1 은 진한색
labels = [s[:14] + "…" for s in samples]  # 축에 다 못 넣으니 앞 14글자
ax.set_xticks(range(len(samples)), labels, rotation=35, ha="right", fontsize=9)  # 아래 눈금
ax.set_yticks(range(len(samples)), labels, fontsize=9)  # 왼쪽 눈금
for i in range(len(samples)):  # 칸마다
    for j in range(len(samples)):  # 숫자도 적어 둔다
        ax.text(  # 칸 한가운데
            j, i, f"{sim[i][j]:.2f}",  # 소수 둘째 자리
            ha="center", va="center", fontsize=9,  # 가운데 정렬
            color="white" if sim[i][j] > 0.6 else "black",  # 진한 칸은 흰 글씨
        )
ax.set_title("문장끼리 얼마나 가까운가")  # 제목
fig.colorbar(im, ax=ax, shrink=0.8)  # 오른쪽 색 막대
fig.tight_layout()  # 글자가 잘리지 않게 여백
fig.savefig("similarity.png", dpi=150)  # day02/similarity.png
print("저장: similarity.png")  # 확인 메시지
plt.show()  # 노트북에 그림을 띄운다


조각 전체를 배치로 임베딩하고 `vectors.npy` 에 저장한다.


In [ ]:
texts = [c["text"] for c in chunks]  # API 에 넣을 본문만 순서대로
t0 = time.time()  # 시작 시각
vecs = []  # 벡터를 순서대로 쌓는다
BATCH = 100  # 한 번에 너무 많이 보내지 않는다
for i in range(0, len(texts), BATCH):  # 0, 100, 200, …
    r = client.embeddings.create(model=EMBED_MODEL, input=texts[i : i + BATCH])  # 이번 묶음
    vecs.extend(d.embedding for d in r.data)  # 응답 순서 = 입력 순서
    print(f"  {min(i + BATCH, len(texts)):>4}/{len(texts)} …")  # 진행
elapsed = time.time() - t0  # 걸린 시간

total_tokens = sum(len(t) for t in texts) // 2  # 대략치. 토큰 수가 아니라 글자/2
print()  # 빈 줄
print(f"조각 {len(vecs):,}개 임베딩 · {elapsed:.1f}초")  # 개수와 지연
print(f"대략 {total_tokens:,}토큰 → text-embedding-3-small 기준 1달러도 안 된다")  # 비용 감각

np.save("vectors.npy", np.array(vecs))  # 넘파이 배열로 디스크에 저장
print("저장: vectors.npy")  # 다음 셀·다음 노트북이 이 파일을 읽는다


저장한 벡터로 검색 함수를 다섯 줄로 만든다. 점수는 순위만 본다.


In [ ]:
chunks = [json.loads(l) for l in Path("chunks.jsonl").read_text(encoding="utf-8").splitlines()]  # 조각 다시 읽기
V = np.load("vectors.npy")  # (조각 수, 차원)
V = V / np.linalg.norm(V, axis=1, keepdims=True)  # 검색 전에 길이를 1로 맞춰 둔다
print(f"조각 {len(chunks):,}개 · 벡터 {V.shape}")  # 두 개수가 같아야 한다


def search(question, k=3):  # 질문 하나, 상위 k 조각
    q = client.embeddings.create(model=EMBED_MODEL, input=[question]).data[0].embedding  # 질문도 같은 모델로
    q = np.array(q) / np.linalg.norm(q)  # 길이 1
    scores = V @ q  # 모든 조각과의 유사도. 1에 가까울수록 가깝다
    top = np.argsort(scores)[::-1][:k]  # 높은 순으로 k개 인덱스
    return [(chunks[i], float(scores[i])) for i in top]  # (조각 dict, 점수)


QUESTION = "어린이 정보를 다룰 때 지켜야 할 것은?"  # 낱말 검색과 같은 질문
for c, s in search(QUESTION):  # 가까운 조각부터
    m = c["metadata"]  # 출처·헤더·chunk_id
    print(f"{s:.3f}  {m['chunk_id']:20s} {m.get('h3') or m['doc_id']}")  # 점수, 아이디, 소제목
    print(f"        {c['text'][:90].strip()}…")  # 본문 앞부분
    print()  # 조각 사이 빈 줄


In [ ]:
for q in [  # 여러 질문을 같은 검색기로
    "정보주체에게는 어떤 권리가 있나?",  # 법
    "실수로 정보가 새어 나갔다면 무엇을 해야 하나?",  # 법
    "밀 재배 의사결정 지원 시스템은 무엇을 하나?",  # 논문
    "대형 언어 모델의 환각이란 무엇인가?",  # 위키
    "미니 프로젝트는 언제 하나?",  # 시간표
]:
    top = search(q, k=1)[0]  # 가장 가까운 조각 하나
    m = top[0]["metadata"]  # 그 조각의 출처
    print(f"Q. {q}")  # 질문
    print(f"   → {top[1]:.3f} {m['doc_id']} · {m.get('h3') or ''}")  # 점수와 문서
    print(f"     {top[0]['text'][:80].strip()}…")  # 본문 맛보기
    print()  # 다음 질문 전에 빈 줄
